# 04 · Link Prediction — Graph Autoencoder (GAE)

**Task:** predict *missing* edges ("who should connect to whom?"). This is **self-supervised** — labels come from
the graph itself. Hide some real edges, train on the rest, then score all node pairs.

**Model — GAE.** An encoder (GCN) maps nodes to embeddings $z_v$; the decoder is parameter-free — the probability
of an edge is the **dot product** of embeddings:

$$ p(u\!\sim\! v) = \sigma(z_u^\top z_v) $$

We optimise binary cross-entropy over real (positive) vs. sampled (negative) edges, and measure **AUC** and
**Average Precision**.

**Source:** Kipf & Welling, *Variational Graph Auto-Encoders*, NeurIPS BDL 2016
([arXiv:1611.07308](https://arxiv.org/abs/1611.07308)).

In [1]:
import sys, os
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # safe Apple-GPU fallback

import torch
from utils.device import get_device, device_report
device = get_device()
print(device_report())


Selected device : MPS
CUDA GPU        : not available
Apple MPS (Metal): available
CPU threads     : 4


In [2]:
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GAE, GCNConv

tf = T.Compose([T.NormalizeFeatures(), T.ToDevice(device),
                T.RandomLinkSplit(num_val=0.05, num_test=0.1, is_undirected=True,
                                  split_labels=True, add_negative_train_samples=False)])
ds = Planetoid(os.path.join(ROOT, "04_link_prediction", "data", "Cora"), name="Cora", transform=tf)
train_data, val_data, test_data = ds[0]
print("train edges:", train_data.edge_index.size(1), "| test pos edges:", test_data.pos_edge_label_index.size(1))

class Encoder(torch.nn.Module):
    def __init__(self, i, h, o):
        super().__init__(); self.c1 = GCNConv(i, h); self.c2 = GCNConv(h, o)
    def forward(self, x, ei): return self.c2(self.c1(x, ei).relu(), ei)

model = GAE(Encoder(ds.num_features, 32, 16)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

@torch.no_grad()
def evaluate(d):
    model.eval(); z = model.encode(d.x, d.edge_index)
    return model.test(z, d.pos_edge_label_index, d.neg_edge_label_index)

for epoch in range(1, 201):
    model.train(); opt.zero_grad()
    z = model.encode(train_data.x, train_data.edge_index)
    loss = model.recon_loss(z, train_data.pos_edge_label_index)
    loss.backward(); opt.step()
    if epoch % 40 == 0:
        auc, ap = evaluate(val_data); print(f"epoch {epoch:3d} | loss {loss:.3f} | val AUC {auc:.3f} AP {ap:.3f}")
auc, ap = evaluate(test_data); print(f"\nTEST AUC {auc:.3f} | AP {ap:.3f}")

train edges: 8976 | test pos edges: 527


epoch  40 | loss 1.027 | val AUC 0.820 AP 0.838


epoch  80 | loss 0.980 | val AUC 0.841 AP 0.859
epoch 120 | loss 0.940 | val AUC 0.876 AP 0.888


epoch 160 | loss 0.902 | val AUC 0.881 AP 0.887
epoch 200 | loss 0.884 | val AUC 0.886 AP 0.891



TEST AUC 0.911 | AP 0.912


## ✅ Notes
- Expect **AUC ~0.90+**. We never used node *labels* — structure alone carries the signal.
- The learned embeddings still cluster by topic; colour a t-SNE of `model.encode(...)` by `Planetoid(...).y` to see it.
- Try PyG's **VGAE** (variational version) for probabilistic embeddings.

➡ You've covered node-, graph-, and edge-level tasks. Build something in `project/`!